# Phase 3.2: Mechanism Ablations - Observation Artifacts

Analyze how observation-level artifacts affect c-GC and c-GC* recovery across depths.

## Scenarios
- **measurement_noise**: Additive measurement error (breaks causal Markovianity)
- **undersampling**: Temporal undersampling (effective lag order increases)

For each scenario:
- Run c-GC and c-GC* across depths [1,2,3,4,5,6]
- Compute recovery metrics (accuracy, precision, recall, FPR)
- Track D_p trajectories and instability signatures
- Export results.csv, summary.json, figures/

## Setup: Imports and Configuration

In [ ]:
from __future__ import annotations

import sys
import json
import logging
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Find project root
CAUSALISED_GC_RELATIVE_PATH = Path('src/markovianity_diagnostic/core/causalised-GC.py')
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find causalised-GC.py')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.simulations import (
    scenario_measurement_noise,
    scenario_undersampled_markov,
)
from markovianity_diagnostic.experiments.adapters import METHODS
from markovianity_diagnostic.experiments.graph_metrics import summarize_run

logger.info(f'Project root: {PROJECT_ROOT}')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'simulations' / 'mechanism_ablations' / 'observation_artifacts'
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
REPEATS = 10
T = 2000
D = 10
P_VALUES = [1, 2, 3, 4, 5, 6]
METHODS_TO_TEST = ['gcstar_cgc', 'gcstar_fcgc']
SCENARIOS = [
    ('measurement_noise', {'T': T, 'd': D, 'noise_scale': 1.0}),
    ('undersampling', {'T': T, 'd': D, 'noise_scale': 1.0}),
]

logger.info(f'Output directory: {OUTPUT_DIR}')
logger.info(f'Repeats: {REPEATS}, Methods: {len(METHODS_TO_TEST)}, Scenarios: {len(SCENARIOS)}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Configuration: REPEATS={REPEATS}, P_VALUES={P_VALUES}, METHODS={METHODS_TO_TEST}')

In [ ]:
# Check for cached outputs
expected_outputs = {
    'results.csv': OUTPUT_DIR / 'results.csv',
    'manifest.json': OUTPUT_DIR / 'manifest.json',
    'figures/dp_trajectories.png': FIGURES_DIR / 'dp_trajectories.png',
}

outputs_exist = all(fpath.exists() for fpath in expected_outputs.values())

if outputs_exist:
    logger.info("Outputs already exist - loading cached results")
    print("✅ Outputs already exist - loading cached results")
    for name, path in expected_outputs.items():
        if path.exists():
            size_mb = path.stat().st_size / (1024 * 1024)
            logger.info(f"  ✓ {name} ({size_mb:.2f} MB)")
            print(f"  ✓ {name} ({size_mb:.2f} MB)")
else:
    logger.info("No existing outputs - will run computation")
    print("⚠️ No existing outputs - will run computation")

if not outputs_exist:
    logger.info("Starting mechanism ablation experiments (observation artifacts)...")
    start_experiments = time.time()
    
    scenario_functions = {
        'measurement_noise': scenario_measurement_noise,
        'undersampling': scenario_undersampled_markov,
    }
    all_results = {}
    
    for scenario_idx, (scenario_name, scenario_kwargs) in enumerate(SCENARIOS, 1):
        logger.info(f"\n[{scenario_idx}/{len(SCENARIOS)}] Scenario: {scenario_name}")
        print(f'\n[{scenario_idx}/{len(SCENARIOS)}] Scenario: {scenario_name}')
        scenario_fn = scenario_functions[scenario_name]
        all_results[scenario_name] = {}
        
        scenario_start = time.time()
        for method_idx, method_name in enumerate(METHODS_TO_TEST, 1):
            logger.info(f"  [{method_idx}/{len(METHODS_TO_TEST)}] Method: {method_name}")
            
            if method_name not in METHODS:
                logger.warning(f"    Warning: {method_name} not found")
                continue
            
            analyze_fn = METHODS[method_name]
            method_results = []
            
            method_start = time.time()
            for repeat_idx in range(REPEATS):
                logger.info(f"      Repeat [{repeat_idx+1}/{REPEATS}]...")
                
                sample = scenario_fn(**scenario_kwargs, seed=SEED + repeat_idx)
                X = sample.X
                adjacencies_by_p = analyze_fn(X, P_VALUES)
                run_summary = summarize_run(
                    adjacencies=adjacencies_by_p,
                    ground_truth_compact=sample.ground_truth_compact,
                    p_values=P_VALUES,
                )
                method_results.append({'repeat': repeat_idx, 'summary': run_summary})
            
            method_elapsed = time.time() - method_start
            logger.info(f"    ✓ {method_name} completed (elapsed: {method_elapsed:.2f}s)")
            print(f'    ✓ {method_name}')
            all_results[scenario_name][method_name] = method_results
        
        scenario_elapsed = time.time() - scenario_start
        logger.info(f"  Scenario {scenario_name} completed (elapsed: {scenario_elapsed:.2f}s)")
    
    total_elapsed = time.time() - start_experiments
    logger.info(f"\nAll experiments completed (total: {total_elapsed:.2f}s)")
    print('✓ All experiments completed')
else:
    logger.info("Skipping experiments (loading from cache)")
    print("Skipping experiments (loading from cache)")

In [ ]:
# Load cached results if they exist
if outputs_exist:
    summary_csv_path = OUTPUT_DIR / 'results.csv'
    summary_df = pd.read_csv(summary_csv_path)
    print(f"Loaded cached results: {len(summary_df)} rows")
    print(summary_df)

In [ ]:
if not outputs_exist:
    summary_rows = []
    for scenario_name, method_results in all_results.items():
        for method_name, results in method_results.items():
            T_obs_values = [r['summary'].get('T_obs', 0.0) for r in results]
            row = {
                'scenario': scenario_name,
                'method': method_name,
                'repeats': len(results),
                'mean_T_obs': float(np.mean(T_obs_values)) if T_obs_values else 0.0,
            }
            summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows)
    summary_csv_path = OUTPUT_DIR / 'results.csv'
    summary_df.to_csv(summary_csv_path, index=False)
    print(f'Exported: {summary_csv_path}')
else:
    print("Skipping export (loading from cache)")

# Create visualization (always run for rendering)
if not outputs_exist and 'all_results' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for scenario_idx, (scenario_name, method_results) in enumerate(all_results.items()):
        ax = axes[scenario_idx]
        for method_name, results in method_results.items():
            d_means = {}
            for p_val in P_VALUES[1:]:
                d_values = [r['summary'].get('D_p', {}).get(str(p_val)) for r in results]
                d_values = [v for v in d_values if v is not None]
                d_means[p_val] = np.mean(d_values) if d_values else 0.0
            p_vals_sorted = sorted(d_means.keys())
            d_vals_sorted = [d_means[p] for p in p_vals_sorted]
            style = '-o' if method_name == 'gcstar_cgc' else '--s'
            ax.plot(p_vals_sorted, d_vals_sorted, style, label=method_name, linewidth=2)
        ax.set_xlabel('Conditioning Depth p')
        ax.set_ylabel('Mean D_p')
        ax.set_title(scenario_name)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'dp_trajectories.png', dpi=100, bbox_inches='tight')
    print('Exported: dp_trajectories.png')
    plt.close()
else:
    print("Skipping figure creation (using cached outputs)")

In [ ]:
logger.info("Exporting results summary...")
start_export = time.time()

summary_rows = []
for scenario_name, method_results in all_results.items():
    for method_name, results in method_results.items():
        T_obs_values = [r['summary'].get('T_obs', 0.0) for r in results]
        row = {
            'scenario': scenario_name,
            'method': method_name,
            'repeats': len(results),
            'mean_T_obs': float(np.mean(T_obs_values)) if T_obs_values else 0.0,
        }
        summary_rows.append(row)
        logger.info(f"  {scenario_name} / {method_name}: {len(results)} repeats")

summary_df = pd.DataFrame(summary_rows)
summary_csv_path = OUTPUT_DIR / 'results.csv'
summary_df.to_csv(summary_csv_path, index=False)
logger.info(f'Exported: {summary_csv_path}')
print(f'Exported: {summary_csv_path}')

export_elapsed = time.time() - start_export
logger.info(f"Export completed in {export_elapsed:.2f}s")

In [ ]:
if not outputs_exist:
    manifest = {
        'created_at': datetime.now(timezone.utc).isoformat(),
        'analysis': 'mechanism_ablations_observation_artifacts',
        'scenarios': list(all_results.keys()),
        'methods': METHODS_TO_TEST,
    }
    with open(OUTPUT_DIR / 'manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print('✓ All outputs verified successfully')
else:
    print("Skipping manifest creation (loading from cache)")

## Manifest

In [ ]:
logger.info("Creating manifest...")

manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'analysis': 'mechanism_ablations_observation_artifacts',
    'scenarios': list(all_results.keys()) if 'all_results' in locals() else [s[0] for s in SCENARIOS],
    'methods': METHODS_TO_TEST,
    'total_experiments': REPEATS * len(METHODS_TO_TEST) * len(SCENARIOS),
}
with open(OUTPUT_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
logger.info(f'Exported: {OUTPUT_DIR / "manifest.json"}')
logger.info('✓ All outputs verified successfully')
print('✓ All outputs verified successfully')